# Surface Mapper Playground

This notebook tests creating a `Surface` object with `ow_to_sid()` using normalized SurfaceGrid test JSON input, then exports a flattened JSON representation including inherited attributes.

### Environment setup and imports

Import standard modules and project models/mappers. Paths are configured relative to the repository root so this notebook can run from the project checkout.

In [ ]:
import json
from pathlib import Path

from pydantic import ValidationError

from dsis_model_sdk.models.common import SurfaceGrid

from mappers.surface_helpers import flatten_dict, normalize_surfacegrid_payload
from mappers.surface_mappers import ow_to_sid
from models.interpretation import PipelineMetadata

In [ ]:
repo_root = Path.cwd()
input_path = repo_root / "../tests/data/surfacegrid_volve_public.json"

### Load normalized SurfaceGrid test JSON

Load test JSON, apply normalization used in tests, and confirm critical keys exist.

In [ ]:
raw_payload = json.loads(input_path.read_text(encoding="utf-8"))
normalized_payload = normalize_surfacegrid_payload(raw_payload)

required_keys = sorted(
    name for name, field in SurfaceGrid.model_fields.items() if field.is_required()
)

missing_required = [key for key in required_keys if key not in normalized_payload]
assert not missing_required, f"Missing required keys: {missing_required}"

surfacegrid = SurfaceGrid.model_validate(normalized_payload)
print("Loaded and validated SurfaceGrid")
print(
    f"native_uid={surfacegrid.native_uid}, name={surfacegrid.map_data_set_name}, crs={surfacegrid.crs}"
)

### Instantiate Surface from `ow_to_sid()`

Create pipeline metadata and run conversion. If mapper validation fails, the error is shown clearly.

In [ ]:
pipeline_metadata = PipelineMetadata(
    id="11111111-1111-1111-1111-111111111111",
    database="BG4FROST",
    project="VOLVE_PUBLIC",
)

In [ ]:
surface = None
try:
    surface = ow_to_sid(surfacegrid, pipeline_metadata)
    print(f"Created object: {type(surface).__name__}")
    print(surface.model_dump_json(indent=2))
except ValidationError as exc:
    print("ow_to_sid validation failed:")
    print(exc)
    raise

### Flatten Surface object to JSON

Flatten nested dictionaries to dot-path keys for deterministic, compact comparison and downstream inspection.

In [ ]:
surface_dict = surface.model_dump(mode="json", exclude_none=False)
flat_surface = dict(sorted(flatten_dict(surface_dict).items(), key=lambda kv: kv[0]))
print(f"Flattened keys: {len(flat_surface)}")
print(json.dumps(dict(list(flat_surface.items())), indent=2))

### Notes

- This notebook uses the same normalization logic from `surface_helpers.py` as tests (normalizes `rotation_i`, `rotation_j`, `data_min`, `data_max`).
